# Commons Dilemma — Results Analysis
**ESADE MIBA Capstone 2026 · Strategic Coherence in LLMs**

All numbers are computed from raw data — no hardcoded conclusions.  
Every estimate includes a 95% confidence interval.

**Conditions**: `undisclosed` · `ai` · `human` · `human_prior`  
**Info conditions**: `full` (pool shown) · `partial` (pool hidden)  
**Risk conditions**: `deterministic` · `probabilistic`  
**Models**: Claude Opus · Claude Sonnet · GPT-4o · GPT-4o-mini · Gemini 2.5 Flash · Gemini 2.5 Flash Lite  
**Rounds per game**: 20  
**Key parameter**: θ — exploitation intensity (extraction / sustainable share)

---
### Sections
1. Setup & Data Loading
2. Data Quality Check
3. Pool Dynamics — Collapse Rate by Condition
4. Exploitation Intensity θ by Condition (with 95% CI)
5. Per-Model θ by Condition
6. Info × Risk Structure Effect (2×2 design)
7. Round-by-Round Extraction Dynamics
8. Belief Calibration β
9. Statistical Tests
10. Key Findings Summary

## 1. Setup & Data Loading

In [ ]:
import os
import glob
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR  = os.path.join(REPO_ROOT, 'all_raw')
print(f'Data dir : {DATA_DIR}')
print(f'Exists   : {os.path.isdir(DATA_DIR)}')

In [ ]:
cd_files = sorted(glob.glob(os.path.join(DATA_DIR, 'cd_*.csv')))
print(f'Found {len(cd_files)} Commons Dilemma CSV files')

frames = []
for f in cd_files:
    df_tmp = pd.read_csv(f)
    df_tmp.columns = df_tmp.columns.str.strip()
    df_tmp['source_file'] = os.path.basename(f)
    frames.append(df_tmp)

raw = pd.concat(frames, ignore_index=True)
print(f'Total rows         : {len(raw):,}')
print(f'Conditions         : {sorted(raw["condition"].unique())}')
print(f'Info conditions    : {sorted(raw["info_condition"].unique())}')
print(f'Risk conditions    : {sorted(raw["risk_condition"].unique())}')
raw.head(3)

## 2. Data Quality Check

In [ ]:
required = ['game_id','condition','info_condition','risk_condition','matchup','round',
            'pool_before_regen','sustainable_share','total_extraction',
            'pool_collapsed','model_1','extraction_1','belief_1',
            'model_2','extraction_2','belief_2']
missing = [c for c in required if c not in raw.columns]
assert not missing, f'Missing columns: {missing}'
print('Schema OK')

# Extraction sanity: must be >= 0
for col in ['extraction_1','extraction_2']:
    neg = (raw[col] < 0).sum()
    print(f'{col} negative values : {neg}')

# Belief range
for col in ['belief_1','belief_2']:
    out = raw[(raw[col] < 0) | (raw[col] > 1)]
    print(f'{col} out of [0,1]    : {len(out)} rows')

# Round counts per file
rc = raw.groupby('source_file')['round'].max()
print(f'\nRound max per file:\n{rc.value_counts().to_string()}')

print('\nRows per condition × info × risk:')
print(raw.groupby(['condition','info_condition','risk_condition']).size().to_string())

In [ ]:
# ── Build long-form (one row per player per round) ────────────────────────────
def to_long(df):
    def side(p):
        o = '2' if p == '1' else '1'
        base = df[[
            'game_id','condition','info_condition','risk_condition',
            'matchup','round','source_file',
            'pool_before_regen','pool_after_regen','sustainable_share',
            'total_extraction','pool_after_extraction','pool_collapsed',
            f'model_{p}', f'extraction_{p}', f'belief_{p}',
            f'payoff_{p}',  f'cumulative_{p}',
            f'extraction_{o}',
        ]].rename(columns={
            f'model_{p}':       'model',
            f'extraction_{p}':  'extraction',
            f'belief_{p}':      'belief',
            f'payoff_{p}':      'payoff',
            f'cumulative_{p}':  'cumulative',
            f'extraction_{o}':  'opp_extraction',
        }).copy()
        base['player'] = p
        return base
    return pd.concat([side('1'), side('2')], ignore_index=True)

long = to_long(raw)

# θ — exploitation intensity: extraction / sustainable_share per player
# sustainable_share = pool * regen_rate / n_players (pre-computed in CSV)
long['theta'] = long['extraction'] / long['sustainable_share'].replace(0, np.nan)
long['over_extract'] = (long['theta'] > 1.0).astype(int)

# Belief calibration: |stated_belief - (opp_extraction > sustainable_share)|
# Opponent over-extracts = 1 (defect-like), sustainable = 0 (cooperate-like)
long['opp_over'] = (long['opp_extraction'] > long['sustainable_share']).astype(float)
long['beta_err'] = (long['belief'] - long['opp_over']).abs()

print(f'Long-form rows : {len(long):,}')
print(f'Models         : {sorted(long["model"].unique())}')
print(f'θ stats  mean={long["theta"].mean():.3f}  median={long["theta"].median():.3f}  '
      f'over-extract rate={long["over_extract"].mean():.1%}')
long.head(3)

## 3. Pool Dynamics — Collapse Rate

In [ ]:
def mean_ci(x, confidence=0.95):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan
    m = x.mean()
    if set(np.unique(x)).issubset({0.0, 1.0}):
        from statsmodels.stats.proportion import proportion_confint
        lo, hi = proportion_confint(int(x.sum()), n, alpha=1-confidence, method='wilson')
        return m, lo, hi
    se = stats.sem(x)
    t  = stats.t.ppf((1 + confidence) / 2, df=n-1)
    return m, m - t*se, m + t*se

# Collapse rate per game (did pool collapse in this game?)
game_collapse = (
    raw.groupby(['source_file','game_id','condition','info_condition','risk_condition','matchup'])
    ['pool_collapsed'].max().reset_index()
)
game_collapse['collapsed'] = (game_collapse['pool_collapsed'] > 0).astype(int)

collapse_stats = []
for (cond, info, risk), grp in game_collapse.groupby(['condition','info_condition','risk_condition']):
    m, lo, hi = mean_ci(grp['collapsed'])
    collapse_stats.append({'condition':cond,'info':info,'risk':risk,
                           'collapse_rate':m,'ci_lo':lo,'ci_hi':hi,'n_games':len(grp)})
collapse_df = pd.DataFrame(collapse_stats)

print('=== Pool Collapse Rate ===')
for _, r in collapse_df.sort_values(['condition','info','risk']).iterrows():
    print(f"  {r['condition']:12s} {r['info']:8s} {r['risk']:15s}  "
          f"{r['collapse_rate']:.1%}  95% CI [{r['ci_lo']:.1%}, {r['ci_hi']:.1%}]  "
          f"(n={r['n_games']} games)")

In [ ]:
# Pool level over rounds (average across all games)
pool_round = (
    raw.groupby(['condition','info_condition','risk_condition','round'])
    ['pool_after_extraction'].agg(['mean','std','count']).reset_index()
)
pool_round['se'] = pool_round['std'] / np.sqrt(pool_round['count'])
t_crit = stats.t.ppf(0.975, df=pool_round['count']-1)
pool_round['ci_lo'] = pool_round['mean'] - t_crit * pool_round['se']
pool_round['ci_hi'] = pool_round['mean'] + t_crit * pool_round['se']

cond_colors = {'undisclosed':'#94a3b8','ai':'#5b8dee','human':'#52c97a','human_prior':'#e8865a'}
info_vals = sorted(pool_round['info_condition'].unique())
risk_vals = sorted(pool_round['risk_condition'].unique())

fig, axes = plt.subplots(len(risk_vals), len(info_vals),
                         figsize=(13, 4.5 * len(risk_vals)), squeeze=False)

for ri, risk in enumerate(risk_vals):
    for ii, info in enumerate(info_vals):
        ax = axes[ri][ii]
        sub = pool_round[(pool_round['info_condition']==info) &
                         (pool_round['risk_condition']==risk)]
        for cond in sorted(sub['condition'].unique()):
            s = sub[sub['condition']==cond].sort_values('round')
            color = cond_colors.get(cond,'#aaa')
            ax.plot(s['round'], s['mean'], label=cond, color=color, linewidth=1.8)
            ax.fill_between(s['round'], s['ci_lo'], s['ci_hi'], color=color, alpha=0.12)
        ax.set_title(f'{info} · {risk}')
        ax.set_xlabel('Round')
        ax.set_ylabel('Pool Size')
        ax.legend(title='Condition', fontsize=8)

fig.suptitle('Pool Level Over Rounds (shaded = 95% CI)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../notebooks/fig_cd_pool_dynamics.png', bbox_inches='tight')
plt.show()

## 4. Exploitation Intensity θ by Condition (with 95% CI)

In [ ]:
# Aggregate at game level to avoid round-level pseudo-replication
game_level = (
    long.groupby(['source_file','game_id','condition','info_condition','risk_condition','model'])
    .agg(
        theta_mean   = ('theta',        'mean'),
        over_rate    = ('over_extract',  'mean'),
        beta_mean    = ('beta_err',      'mean'),
        n_rounds     = ('round',         'count'),
        collapse     = ('pool_collapsed','max'),
    ).reset_index()
)

theta_stats = []
for (cond, info, risk), grp in game_level.groupby(['condition','info_condition','risk_condition']):
    m, lo, hi = mean_ci(grp['theta_mean'])
    theta_stats.append({'condition':cond,'info':info,'risk':risk,
                        'mean':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
theta_df = pd.DataFrame(theta_stats)

print('=== θ (Exploitation Intensity) — θ=1.0 is exactly sustainable ===')
for _, r in theta_df.sort_values(['condition','info','risk']).iterrows():
    flag = '⚠ over' if r['mean'] > 1.0 else '✓ sustain'
    print(f"  {r['condition']:12s} {r['info']:8s} {r['risk']:15s}  "
          f"θ={r['mean']:.3f}  95% CI [{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]  "
          f"{flag}  (n={r['n']})")

In [ ]:
fig, axes = plt.subplots(1, len(risk_vals), figsize=(7 * len(risk_vals), 5), squeeze=False)
cond_colors = {'undisclosed':'#94a3b8','ai':'#5b8dee','human':'#52c97a','human_prior':'#e8865a'}

for ri, risk in enumerate(risk_vals):
    ax = axes[0][ri]
    sub = theta_df[theta_df['risk']==risk]
    conds = sorted(sub['condition'].unique())
    infos = sorted(sub['info'].unique())
    x = np.arange(len(conds))
    width = 0.35
    info_hatches = {'full':'', 'partial':'///'}
    for ii, info in enumerate(infos):
        s = sub[sub['info']==info].set_index('condition').reindex(conds)
        offset = (ii - (len(infos)-1)/2) * width
        bars = ax.bar(x + offset, s['mean'].fillna(0), width,
                      color=[cond_colors.get(c,'#aaa') for c in conds],
                      alpha=0.85, hatch=info_hatches.get(info,''),
                      label=f'{info} info', edgecolor='white')
        ax.errorbar(x + offset, s['mean'].fillna(0),
                    yerr=[(s['mean']-s['ci_lo']).fillna(0),
                          (s['ci_hi']-s['mean']).fillna(0)],
                    fmt='none', color='#222', capsize=5, linewidth=1.4)
    ax.axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='θ=1 (sustainable)')
    ax.set_xticks(x)
    ax.set_xticklabels(conds, rotation=15)
    ax.set_ylabel('θ (extraction / sustainable share)')
    ax.set_title(f'θ by Condition — {risk} risk\n(error bars = 95% CI)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../notebooks/fig_cd_theta_condition.png', bbox_inches='tight')
plt.show()

## 5. Per-Model θ by Condition

In [ ]:
def mcolor(name):
    n = name.lower()
    if 'claude' in n: return '#8b5cf6'
    if 'gpt'    in n: return '#10b981'
    if 'gemini' in n: return '#60a5fa'
    return '#888'

model_theta = []
for (model, cond), grp in game_level.groupby(['model','condition']):
    m, lo, hi = mean_ci(grp['theta_mean'])
    model_theta.append({'model':model,'condition':cond,
                        'mean':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
model_theta_df = pd.DataFrame(model_theta)

conds = sorted(model_theta_df['condition'].unique())
models_list = sorted(model_theta_df['model'].unique())
x = np.arange(len(models_list))
width = 0.7 / len(conds)

fig, ax = plt.subplots(figsize=(13, 5))
cond_colors2 = {'undisclosed':'#94a3b8','ai':'#5b8dee','human':'#52c97a','human_prior':'#e8865a'}

for j, cond in enumerate(conds):
    sub = model_theta_df[model_theta_df['condition']==cond].set_index('model').reindex(models_list)
    offset = (j - (len(conds)-1)/2) * width
    ax.bar(x + offset, sub['mean'].fillna(0), width,
           color=cond_colors2.get(cond,'#aaa'), alpha=0.85, label=cond)
    ax.errorbar(x + offset, sub['mean'].fillna(0),
                yerr=[(sub['mean']-sub['ci_lo']).fillna(0),
                      (sub['ci_hi']-sub['mean']).fillna(0)],
                fmt='none', color='#222', capsize=4, linewidth=1.2)

ax.axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='θ=1 (sustainable)')
ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=20, ha='right')
ax.set_ylabel('θ (exploitation intensity)')
ax.set_title('Per-Model θ by Condition\n(error bars = 95% CI, game-level aggregation)')
ax.legend(title='Condition', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('../notebooks/fig_cd_model_theta.png', bbox_inches='tight')
plt.show()

## 6. Info × Risk Structure Effect (2×2 design)

In [ ]:
cell_stats = []
for (info, risk), grp in game_level.groupby(['info_condition','risk_condition']):
    m_theta, lo_t, hi_t = mean_ci(grp['theta_mean'])
    m_over,  lo_o, hi_o = mean_ci(grp['over_rate'])
    m_col,   lo_c, hi_c = mean_ci(grp['collapse'].clip(upper=1))
    cell_stats.append({
        'info':info,'risk':risk,
        'theta':m_theta,'theta_lo':lo_t,'theta_hi':hi_t,
        'over_rate':m_over,'over_lo':lo_o,'over_hi':hi_o,
        'collapse':m_col,'col_lo':lo_c,'col_hi':hi_c,
        'n':len(grp)
    })
cell_df = pd.DataFrame(cell_stats)

print('=== 2×2 Structure Effect ===')
print(f'{"":30s} {"θ":>10s}  {"Over-extract %":>15s}  {"Collapse %":>12s}  n')
for _, r in cell_df.sort_values(['info','risk']).iterrows():
    print(f"  {r['info']:8s} × {r['risk']:15s}  "
          f"θ={r['theta']:.3f} [{r['theta_lo']:.3f},{r['theta_hi']:.3f}]  "
          f"over={r['over_rate']:.1%}  "
          f"collapse={r['collapse']:.1%}  "
          f"n={r['n']}")

In [ ]:
# Heatmap of θ across the 2×2 design
pivot = cell_df.pivot(index='risk', columns='info', values='theta')

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(pivot.values, cmap='RdYlGn_r', vmin=0.8, vmax=1.4, aspect='auto')
plt.colorbar(im, ax=ax, label='θ (exploitation intensity)')
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns)
ax.set_yticklabels(pivot.index)
ax.set_xlabel('Info condition')
ax.set_ylabel('Risk condition')
ax.set_title('Mean θ by Info × Risk\n(red = over-extraction, green = sustainable)')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i,j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                    fontsize=12, fontweight='bold', color='black')
plt.tight_layout()
plt.savefig('../notebooks/fig_cd_2x2_heatmap.png', bbox_inches='tight')
plt.show()

## 7. Round-by-Round Extraction Dynamics

In [ ]:
round_theta = []
for (cond, info, risk, rnd), grp in long.groupby(['condition','info_condition','risk_condition','round']):
    m, lo, hi = mean_ci(grp['theta'].dropna())
    round_theta.append({'condition':cond,'info':info,'risk':risk,'round':rnd,
                        'mean':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
round_theta_df = pd.DataFrame(round_theta)

for risk in risk_vals:
    for info in info_vals:
        sub = round_theta_df[(round_theta_df['risk']==risk) & (round_theta_df['info']==info)]
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(12, 5))
        for cond in sorted(sub['condition'].unique()):
            s = sub[sub['condition']==cond].sort_values('round')
            color = cond_colors.get(cond,'#aaa')
            ax.plot(s['round'], s['mean'], marker='o', markersize=3,
                    label=cond, color=color, linewidth=1.8)
            ax.fill_between(s['round'], s['ci_lo'], s['ci_hi'], color=color, alpha=0.12)
        ax.axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='θ=1 (sustainable)')
        ax.set_xlabel('Round')
        ax.set_ylabel('θ (exploitation intensity)')
        ax.set_title(f'θ Over Rounds — {info} info · {risk} risk\n(shaded = 95% CI)')
        ax.legend(title='Condition')
        plt.tight_layout()
        plt.savefig(f'../notebooks/fig_cd_round_{info}_{risk}.png', bbox_inches='tight')
        plt.show()

## 8. Belief Calibration β

In [ ]:
beta_rows = []
for (model, cond), grp in game_level.groupby(['model','condition']):
    m, lo, hi = mean_ci(grp['beta_mean'])
    beta_rows.append({'model':model,'condition':cond,
                      'beta':m,'ci_lo':lo,'ci_hi':hi,'n':len(grp)})
beta_df = pd.DataFrame(beta_rows)

models_list = sorted(beta_df['model'].unique())
conds = sorted(beta_df['condition'].unique())
x = np.arange(len(models_list))
width = 0.7 / len(conds)

fig, ax = plt.subplots(figsize=(13, 5))
for j, cond in enumerate(conds):
    sub = beta_df[beta_df['condition']==cond].set_index('model').reindex(models_list)
    offset = (j - (len(conds)-1)/2) * width
    ax.bar(x + offset, sub['beta'].fillna(0), width,
           color=cond_colors2.get(cond,'#aaa'), alpha=0.85, label=cond)
    ax.errorbar(x + offset, sub['beta'].fillna(0),
                yerr=[(sub['beta']-sub['ci_lo']).fillna(0),
                      (sub['ci_hi']-sub['beta']).fillna(0)],
                fmt='none', color='#222', capsize=4, linewidth=1.2)

ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=20, ha='right')
ax.set_ylabel('β (Belief Calibration MAE — lower is better)')
ax.set_title('Belief Calibration by Model and Condition\n(error bars = 95% CI)')
ax.legend(title='Condition', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('../notebooks/fig_cd_beta.png', bbox_inches='tight')
plt.show()

## 9. Statistical Tests

In [ ]:
print('=== Kruskal-Wallis: θ across Conditions ===')
groups = [grp['theta_mean'].dropna().values
          for _, grp in game_level.groupby('condition') if len(grp) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f'  H={stat:.3f},  p={p:.4f}')

print('\n=== Kruskal-Wallis: θ across Models ===')
groups = [grp['theta_mean'].dropna().values
          for _, grp in game_level.groupby('model') if len(grp) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f'  H={stat:.3f},  p={p:.4f}')

print('\n=== Kruskal-Wallis: θ across Info conditions ===')
groups = [grp['theta_mean'].dropna().values
          for _, grp in game_level.groupby('info_condition') if len(grp) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f'  H={stat:.3f},  p={p:.4f}')

print('\n=== Kruskal-Wallis: θ across Risk conditions ===')
groups = [grp['theta_mean'].dropna().values
          for _, grp in game_level.groupby('risk_condition') if len(grp) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f'  H={stat:.3f},  p={p:.4f}')

In [ ]:
print('=== Pairwise Mann-Whitney U (conditions, Holm-Bonferroni) ===')
cond_list = sorted(game_level['condition'].unique())
pairs = list(itertools.combinations(cond_list, 2))
raw_p, labels = [], []
for c1, c2 in pairs:
    g1 = game_level[game_level['condition']==c1]['theta_mean'].dropna()
    g2 = game_level[game_level['condition']==c2]['theta_mean'].dropna()
    if len(g1) < 3 or len(g2) < 3:
        continue
    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
    raw_p.append(p)
    labels.append(f'{c1} vs {c2}')
if raw_p:
    reject, p_adj, _, _ = multipletests(raw_p, method='holm')
    for label, p_raw, p_c, rej in zip(labels, raw_p, p_adj, reject):
        sig = '***' if p_c<0.001 else '**' if p_c<0.01 else '*' if p_c<0.05 else 'ns'
        print(f'  {label:35s}  p_raw={p_raw:.4f}  p_adj={p_c:.4f}  {sig}')

## 10. Key Findings Summary

In [ ]:
print('=' * 65)
print('KEY FINDINGS — Commons Dilemma')
print('=' * 65)

print('\n[DATASET]')
print(f'  Total runs      : {len(cd_files)}')
print(f'  Total rounds    : {len(raw):,}')
print(f'  Conditions      : {", ".join(sorted(raw["condition"].unique()))}')
print(f'  Info conditions : {", ".join(sorted(raw["info_condition"].unique()))}')
print(f'  Risk conditions : {", ".join(sorted(raw["risk_condition"].unique()))}')
print(f'  Models          : {", ".join(sorted(long["model"].unique()))}')

print('\n[θ BY CONDITION (game-level, all info/risk combined)]')
for cond, grp in game_level.groupby('condition'):
    m, lo, hi = mean_ci(grp['theta_mean'].dropna())
    flag = '⚠ over-extract' if m > 1.0 else '✓ sustainable'
    print(f'  {cond:15s}  θ={m:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  {flag}')

print('\n[COLLAPSE RATE BY CONDITION]')
for cond, grp in game_collapse.groupby('condition'):
    m, lo, hi = mean_ci(grp['collapsed'])
    print(f'  {cond:15s}  {m:.1%}  95% CI [{lo:.1%}, {hi:.1%}]')

print('\n[θ BY MODEL (all conditions combined)]')
for model, grp in game_level.groupby('model'):
    m, lo, hi = mean_ci(grp['theta_mean'].dropna())
    m_beta, *_ = mean_ci(grp['beta_mean'].dropna())
    print(f'  {model:30s}  θ={m:.3f} [{lo:.3f},{hi:.3f}]  β={m_beta:.3f}')

print('\n[NOTE] θ=1.0 = exactly sustainable. θ>1.0 = over-extraction.')
print('       95% CIs: t-interval (continuous). Tests: Kruskal-Wallis + Holm-Bonferroni.')